## Model Testing

Testing how well this model performs as a standalone unit, procedures:

1. yFinance API used to gather current mixed-cap stock data
2. import the IVA model, feed the data, let it forecast valuations
3. prediction checks will be performed on July 2nd
4. real asset movement and prediction comparisons will be graphed
5. evaluation results will be added to the model card

#### Imports

In [30]:
import pandas as pd
import yfinance as yf
import joblib

#### Data Gathering

In [ ]:
# manual process, tickers need constant updates
# csv file may be different in the future or missing
data = []
tickers = ["JKHY", "FRT", "COP", "GLW", "UBER", "DE", "CRM", 
           "APH", "PFE", "ETN", "BX", "HON", "PLD", "CB", "VRT"]

for ticker in tickers:
    stock = yf.Ticker(ticker)
    info  = stock.info
    price = info.get("currentPrice")
    earnings = info.get("trailingEps")
    growth = info.get("earningsGrowth")
    pe_ratio = info.get("trailingPE")
    data.append({
        "ticker": ticker, "price": price,
        "earnings": earnings, "pe_ratio": pe_ratio,
        "growth": growth, "corp_yield": 2.65
    })

df = pd.DataFrame(data)
df.to_csv("testing_dataset.csv", index=False, mode='a', header=False)
print("Dataset created and saved")

Dataset created and saved


#### Updating Dataframe with Model Predictions

In [ ]:
df = pd.read_csv("testing_dataset.csv")
df.dropna(inplace=True)
df.drop_duplicates(subset=['ticker'], keep='first', inplace=True)

def calculate_intrinsic_value(row):
    cagr = row['growth'] * 100
    iv = (row['earnings'] * (8.5 + (2 * cagr)) * 4.4) / 2.67
    return float(round(iv, 2))

df['iv'] = df.apply(calculate_intrinsic_value, axis=1)
display(df.head())

,ticker,price,earnings,pe_ratio,growth,corp_yield,iv
0,NVDA,220.61,4.90,45.022450,0.956,2.65,1612.56
1,AAPL,298.97,8.25,36.238790,0.218,2.65,708.33
2,MSFT,417.42,16.77,24.890877,0.234,2.65,1528.27
3,AMZN,259.34,8.20,31.626830,0.748,2.65,2136.42
4,GOOGL,387.66,13.10,29.592365,0.820,2.65,3723.93


In [31]:
# loading the model and standard scaler used while training
# applying scaler to the features before feeding into the model
# crucial step since it scales current data to training conditions
artifacts = joblib.load('iv_analyzer.joblib')
scaler = artifacts['scaler']
nn_model = artifacts['model']

classifier_features = ['pe_ratio', 'iv', 'price']
x = df[classifier_features]
x_scaled = scaler.transform(x)

df['prediction'] = nn_model.predict(x_scaled)
probabilities = nn_model.predict_proba(x_scaled)
df['prob_for_1'] = probabilities[:, 1]
df['prob_for_0'] = probabilities[:, 0]
df.to_csv("testing_dataset.csv", index=False)
print("Model testing successful")

Model testing successful


#### To be Updated Later...

Waiting till July 2nd to see if the predictions were true. How much profit or loss would be incurred if 100 shares of each ticker were purchased today based on model predictions.